# SI10-2026 | Ponderada | Análise de Sensibilidade em Métricas de Interface Digital




Nesta atividade, você vai analisar quais variáveis de uma interface digital têm maior impacto sobre a taxa de conversão.

A entrega deve ser feita neste notebook, com código, tabelas, gráficos e respostas curtas.

## Contexto

Uma equipe de produto quer decidir qual métrica de interface deve receber prioridade no próximo ciclo de melhoria.

Os dados representam observações diárias de um aplicativo de compras.

A métrica alvo é a taxa de conversão.

As variáveis de entrada são taxa de abandono do carrinho, profundidade média de scroll e tempo até o primeiro clique em produto.

## Preparação

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.precision", 3)

features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"


## Dados

Execute a célula abaixo para criar a base da atividade.

In [ ]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

,data,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
0,2026-01-01,50.438,77.672,6.664,5.589
1,2026-01-02,39.680,64.633,7.843,6.042
2,2026-01-03,54.004,57.069,9.200,5.318
3,2026-01-04,55.525,75.275,4.671,5.944
4,2026-01-05,32.392,67.145,6.725,6.804


## Parte 1: Exploração

Crie ao menos um gráfico ou tabela para investigar a relação entre as variáveis de entrada e a taxa de conversão.

In [ ]:
# Análise exploratória: correlação entre as variáveis numéricas.
# A correlação ajuda a enxergar se a relação tende a ser positiva, negativa ou fraca.

colunas_numericas = features + [target]

matriz_correlacao = df[colunas_numericas].corr()

correlacao_com_conversao = (
    matriz_correlacao[target]
    .drop(target)
    .sort_values(key=lambda serie: serie.abs(), ascending=False)
    .to_frame("correlação com a taxa de conversão")
)

display(matriz_correlacao)
correlacao_com_conversao


,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
taxa_abandono_carrinho_pct,1.000,-0.068,-0.116,-0.643
profundidade_scroll_pct,-0.068,1.000,0.050,0.485
tempo_primeiro_clique_s,-0.116,0.050,1.000,-0.229
taxa_conversao_pct,-0.643,0.485,-0.229,1.000


,correlação com a taxa de conversão
taxa_abandono_carrinho_pct,-0.643
profundidade_scroll_pct,0.485
tempo_primeiro_clique_s,-0.229


In [ ]:
# Gráfico exploratório: escolhi uma variável de entrada para visualizar contra a conversão.
# Usei o gráfico de dispersão porque ele mostra a direção geral da relação e também a variação dos pontos.

variavel_x = "profundidade_scroll_pct"

if variavel_x not in features:
    raise ValueError("Preencha variavel_x com uma variável da lista features.")

fig = px.scatter(
    df,
    x=variavel_x,
    y=target,
    title=f"Relação entre {variavel_x} e taxa de conversão",
    labels={
        variavel_x: variavel_x.replace("_", " "),
        target: "taxa de conversão (%)",
    },
)
fig.show()


In [ ]:
# Gráfico 2: relação entre profundidade média de scroll e taxa de conversão.
# Esse gráfico complementa o primeiro, porque mostra uma variável com relação positiva com a conversão.

variavel_x_2 = "profundidade_scroll_pct"

fig = px.scatter(
    df,
    x=variavel_x_2,
    y=target,
    title=f"Relação entre {variavel_x_2} e taxa de conversão",
    labels={
        variavel_x_2: "profundidade média de scroll (%)",
        target: "taxa de conversão (%)",
    },
)

fig.show()

Escreva quais duas variáveis você escolheu para a análise de sensibilidade e justifique com evidências da exploração.


**Resposta**:

Escolhi **taxa de abandono do carrinho** e profundidade média de scroll para a análise de sensibilidade.

A escolha veio da tabela de correlação e dos dois gráficos exploratórios. No primeiro gráfico, a relação entre taxa de abandono do carrinho e taxa de conversão mostra uma tendência negativa: quando o abandono aumenta, a conversão tende a diminuir. Isso também aparece na correlação, que ficou em aproximadamente -0,643, sendo a relação mais forte da análise.

No segundo gráfico, a profundidade média de scroll apresenta uma tendência positiva com a taxa de conversão. Ou seja, nos dias em que os usuários rolaram mais a página, a conversão tendeu a ser maior. Essa relação também aparece na correlação, que ficou em aproximadamente +0,485.

Eu deixei de fora o tempo até o primeiro clique porque ele teve uma correlação menor, de aproximadamente -0,229. Ele ainda pode ter influência na experiência do usuário, mas, olhando para esta base, a taxa de abandono e a profundidade de scroll parecem explicar melhor a variação da conversão.


**Documentação Parte 1**

Nesta parte eu usei uma tabela de correlação e dois gráficos de dispersão para entender melhor a relação entre as variáveis de entrada e a taxa de conversão. O primeiro gráfico compara a taxa de abandono do carrinho com a conversão e ajuda a visualizar uma relação negativa. O segundo compara a profundidade média de scroll com a conversão e mostra uma relação positiva. Com isso, a escolha das duas variáveis para a análise de sensibilidade ficou baseada tanto nos números da correlação quanto na visualização dos dados.


## Parte 2: Modelo

Ajuste o modelo abaixo para estimar a taxa de conversão a partir das variáveis de entrada.

In [ ]:
# Modelo linear simples com mínimos quadrados.
# A ideia é estimar a taxa de conversão usando as três variáveis de entrada.

X = df[features].to_numpy()
y = df[target].to_numpy()

# Adiciono uma coluna de 1 para estimar o intercepto do modelo.
X_design = np.column_stack([np.ones(len(X)), X])

coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))

tabela_coeficientes = pd.DataFrame({
    "termo": ["intercepto"] + features,
    "coeficiente": coeficientes,
})

metricas_modelo = pd.DataFrame({
    "métrica": ["MAE", "RMSE"],
    "valor": [mae, rmse],
})

display(tabela_coeficientes)
metricas_modelo


,termo,coeficiente
0,intercepto,7.883
1,taxa_abandono_carrinho_pct,-0.060
2,profundidade_scroll_pct,0.024
3,tempo_primeiro_clique_s,-0.094


,métrica,valor
0,MAE,0.276
1,RMSE,0.344


Interprete o erro do modelo em relação à taxa de conversão.

**Resposta:**

O modelo teve **MAE de aproximadamente 0,276** e **RMSE de aproximadamente 0,344**. Como a taxa de conversão está em pontos percentuais, isso quer dizer que o erro médio absoluto ficou perto de **0,28 ponto percentual**.

Para esta atividade, considero o erro aceitável, porque a base foi gerada com ruído e mesmo assim o modelo conseguiu prever a conversão com uma diferença relativamente pequena. O RMSE é um pouco maior que o MAE porque ele pesa mais os erros maiores, mas ainda ficou em um valor baixo para a escala da conversão.


**Documentação Parte 2**

Aqui eu ajustei uma regressão linear por mínimos quadrados. O modelo tenta encontrar os coeficientes que melhor aproximam a taxa de conversão a partir das variáveis de entrada. Depois comparei a previsão com o valor real usando MAE e RMSE, que são métricas de erro: quanto menores, melhor o ajuste.


## Parte 3: Análise de Sensibilidade

Calcule a sensibilidade para duas variáveis de entrada usando uma variação de 10%.

Use a fórmula: sensibilidade igual à variação percentual da saída dividida pela variação percentual da entrada.

In [ ]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)


linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)

linha_base, saida_base

({'taxa_abandono_carrinho_pct': 47.54587889307049,
  'profundidade_scroll_pct': 62.44918010647032,
  'tempo_primeiro_clique_s': 6.9708957453209095},
 5.868747841831934)

In [ ]:
# Variáveis escolhidas a partir da exploração.
# A variação usada foi de +10% em cada variável, mantendo as outras constantes.

variaveis_escolhidas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
]

if len(variaveis_escolhidas) != 2:
    raise ValueError("Preencha variaveis_escolhidas com duas variáveis da lista features.")

variaveis_invalidas = [v for v in variaveis_escolhidas if v not in features]

if variaveis_invalidas:
    raise ValueError(f"Variáveis fora de features: {variaveis_invalidas}")

variacao_entrada = 0.10

resultados = []

for variavel in variaveis_escolhidas:
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)
    linha_cenario[variavel] = valor_alterado

    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / variacao_entrada

    resultados.append({
        "variável": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saída_original": saida_base,
        "saída_nova": saida_nova,
        "variação_saida_pct": variacao_saida * 100,
        "índice_sensibilidade": indice_sensibilidade,
    })

tabela_sensibilidade = pd.DataFrame(resultados)
tabela_sensibilidade


,variável,valor_original,valor_alterado,saída_original,saída_nova,variação_saida_pct,índice_sensibilidade
0,taxa_abandono_carrinho_pct,47.546,52.300,5.869,5.582,-4.891,-0.489
1,profundidade_scroll_pct,62.449,68.694,5.869,6.020,2.578,0.258


Compare os índices de sensibilidade e indique qual variável tem maior impacto sobre a taxa de conversão.

Mostre o raciocínio: cite os valores da tabela e explique o que eles significam para a decisão.

**Resposta:**

Comparando os índices, a variável com maior impacto foi a **taxa de abandono do carrinho**.

Quando a taxa de abandono aumenta 10%, a taxa de conversão prevista cai de aproximadamente **5,869%** para **5,582%**. Isso representa uma variação de saída de cerca de **-4,891%** e um índice de sensibilidade de aproximadamente **-0,489**.

Já quando a profundidade de scroll aumenta 10%, a conversão prevista sobe de aproximadamente **5,869%** para **6,020%**. A variação de saída fica em torno de **+2,578%**, com índice de sensibilidade de aproximadamente **+0,258**.

Então, apesar de as duas variáveis importarem, o abandono do carrinho tem maior efeito absoluto sobre a conversão. O sinal negativo também é importante: reduzir abandono deve aumentar a conversão, enquanto aumentar scroll tende a ajudar.


**Documentação Parte 3**

A sensibilidade foi calculada criando um cenário artificial: aumentei uma variável em 10% e mantive as outras iguais. Depois comparei quanto a saída mudou proporcionalmente. O índice mostra o quanto a conversão reage a uma mudança percentual na variável de entrada. Quanto maior o valor absoluto do índice, maior o impacto.


## Parte 4: Decisão

Recomende uma ação de produto ou interface com base na análise.

Sua recomendação deve citar os números da tabela de sensibilidade.

**Resposta:**

Eu recomendaria começar pelo abandono do carrinho, porque ele teve o maior índice de sensibilidade em valor absoluto: cerca de **-0,489**. Isso significa que pioras nessa métrica afetam bastante a conversão. A profundidade de scroll também ajuda, com índice perto de **+0,258**, mas o efeito foi menor.

Em termos de interface, minha ação seria simplificar a jornada depois que o produto entra no carrinho: reduzir distrações, deixar custos claros, evitar surpresas de frete no final e melhorar a visibilidade do botão de finalizar compra. Como segundo passo, eu mexeria na página de listagem/produto para facilitar que o usuário encontre informações relevantes sem precisar “caçar” demais pela tela.


Aponte uma limitação, risco ou hipótese da sua análise.

**Resposta:**

O principal risco é tratar a previsão como se fosse uma certeza. O modelo é útil para priorização, mas depende dos dados gerados e de uma relação linear entre as variáveis. Na vida real, a conversão pode ser afetada por fatores que não aparecem aqui, como promoções, preço, qualidade do tráfego, dia da semana, campanhas pagas e problemas técnicos.


**Documentação Parte 4**

Nesta parte eu transformei o resultado numérico em uma decisão de produto. O índice de sensibilidade serviu como critério de priorização: a variável com maior impacto absoluto deveria receber atenção primeiro. Também apontei uma limitação para deixar claro que o modelo apoia a decisão, mas não substitui validação com experimento ou dados reais adicionais.


## Ao Além dos Aléns

Faça uma simulação de Monte Carlo para estimar como a taxa de conversão pode variar sob incerteza nas variáveis de entrada.

In [ ]:
# Simulação de Monte Carlo.
# Em vez de usar apenas uma linha média, vou gerar 1000 cenários possíveis com pequenas variações nas entradas.

n_simulacoes = 1000
rng_monte_carlo = np.random.default_rng(123)

amostras = pd.DataFrame({
    "taxa_abandono_carrinho_pct": rng_monte_carlo.normal(
        linha_base["taxa_abandono_carrinho_pct"], 5, n_simulacoes
    ).clip(25, 75),
    "profundidade_scroll_pct": rng_monte_carlo.normal(
        linha_base["profundidade_scroll_pct"], 8, n_simulacoes
    ).clip(25, 95),
    "tempo_primeiro_clique_s": rng_monte_carlo.normal(
        linha_base["tempo_primeiro_clique_s"], 1.5, n_simulacoes
    ).clip(2, 15),
})

amostras_design = np.column_stack([
    np.ones(len(amostras)),
    amostras[features].to_numpy(),
])
previsoes = amostras_design @ coeficientes

resumo_monte_carlo = pd.Series(previsoes).describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
).to_frame("taxa_conversao_pct_prevista")

riscos = pd.DataFrame({
    "indicador": [
        "probabilidade de conversão abaixo de 5,5%",
        "probabilidade de ficar abaixo da saída base",
    ],
    "valor": [
        np.mean(previsoes < 5.5),
        np.mean(previsoes < saida_base),
    ],
})

display(resumo_monte_carlo)
riscos


,taxa_conversao_pct_prevista
count,1000.000
mean,5.864
std,0.385
min,4.723
10%,5.379
25%,5.601
50%,5.859
75%,6.123
90%,6.353
max,7.313


,indicador,valor
0,"probabilidade de conversão abaixo de 5,5%",0.173
1,probabilidade de ficar abaixo da saída base,0.509


In [ ]:
fig = px.histogram(
    pd.DataFrame({"taxa_conversao_pct_prevista": previsoes}),
    x="taxa_conversao_pct_prevista",
    nbins=30,
    title="Distribuição simulada da taxa de conversão",
)
fig.show()

Interprete o que a distribuição simulada indica sobre o risco da sua recomendação.

**Resposta:**

A distribuição simulada mostra que a conversão prevista fica concentrada perto de **5,86%**, mas existe uma variação natural entre os cenários. O percentil 10 ficou perto de **5,38%** e o percentil 90 ficou perto de **6,35%**, então a maior parte dos cenários simulados aparece dentro dessa faixa.

Isso indica que a recomendação tem potencial, mas também tem risco. A simulação estimou cerca de **17%** de chance de a conversão ficar abaixo de **5,5%**. Por isso, eu não trataria a recomendação como uma aposta única e definitiva: eu aplicaria a mudança primeiro em teste A/B ou rollout gradual, acompanhando se o abandono realmente cai e se a conversão melhora.


**Documentação curta do Monte Carlo**

Monte Carlo é uma simulação com vários cenários possíveis. Aqui eu variei as entradas em torno dos valores médios e usei o modelo para prever a conversão em cada cenário. O histograma ajuda a ver a faixa provável de resultados, e não apenas um número único.


## Além dos Aléns dos Aléns

Como extra, fiz uma leitura de meta: quanto eu precisaria melhorar uma variável para buscar um ganho pequeno, na taxa de conversão?


In [ ]:
# Meta de melhoria.
# Aqui uso o coeficiente do modelo para estimar quanto seria necessário reduzir o abandono
# para tentar ganhar 0,25 ponto percentual de conversão.

meta_ganho_pp = 0.25

coef_abandono = tabela_coeficientes.loc[
    tabela_coeficientes["termo"] == "taxa_abandono_carrinho_pct",
    "coeficiente",
].iloc[0]

reducao_abandono_necessaria_pp = meta_ganho_pp / abs(coef_abandono)
reducao_abandono_necessaria_relativa = (
    reducao_abandono_necessaria_pp / linha_base["taxa_abandono_carrinho_pct"]
) * 100

plano_meta = pd.DataFrame({
    "meta": ["aumentar conversão em 0,25 p.p."],
    "variável usada": ["taxa_abandono_carrinho_pct"],
    "redução necessária no abandono (p.p.)": [reducao_abandono_necessaria_pp],
    "redução necessária relativa (%)": [reducao_abandono_necessaria_relativa],
})

plano_meta


,meta,variável usada,redução necessária no abandono (p.p.),redução necessária relativa (%)
0,"aumentar conversão em 0,25 p.p.",taxa_abandono_carrinho_pct,4.141,8.709


In [ ]:
# Ranking de sensibilidade das três variáveis, só como apoio extra.
# A atividade pedia duas variáveis, mas o ranking completo ajuda a justificar a priorização.

ranking_sensibilidade = []

for variavel in features:
    linha_cenario = linha_base.copy()
    linha_cenario[variavel] = linha_cenario[variavel] * 1.10

    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice = variacao_saida / 0.10

    ranking_sensibilidade.append({
        "variável": variavel,
        "índice_sensibilidade": indice,
        "impacto_absoluto": abs(indice),
    })

ranking_sensibilidade = pd.DataFrame(ranking_sensibilidade).sort_values(
    "impacto_absoluto",
    ascending=False,
)

ranking_sensibilidade


,variável,índice_sensibilidade,impacto_absoluto
0,taxa_abandono_carrinho_pct,-0.489,0.489
1,profundidade_scroll_pct,0.258,0.258
2,tempo_primeiro_clique_s,-0.112,0.112






**Leitura do alem dos alens dos alens**

Para tentar ganhar **0,25 ponto percentual** de conversão apenas reduzindo abandono, o modelo sugere uma redução de aproximadamente **4,14 pontos percentuais** na taxa média de abandono, o que equivale a cerca de **8,71%** de redução relativa sobre a média atual. Isso torna a recomendação mais concreta, porque transforma o resultado do modelo em uma meta de produto mais fácil de acompanhar.


**Documentação curta do alem dos alens dos alens**

Usei o coeficiente da taxa de abandono para transformar uma meta de conversão em uma meta operacional. Como o coeficiente do abandono é negativo, reduzir abandono aumenta a previsão de conversão. O ranking das três sensibilidades foi incluído apenas como apoio, para mostrar que a priorização continua fazendo sentido mesmo olhando todas as variáveis.


## Política de Uso de IA

O uso de IA é permitido para apoio técnico, revisão de texto e estudo dos conceitos.

As escolhas de variáveis, os cálculos, a comparação dos índices e a recomendação devem refletir sua análise dos resultados deste notebook.

Você deve ser capaz de explicar qualquer resposta entregue.

Respostas sem relação com os números gerados, com indícios de cópia ou que não possam ser justificadas poderão ser tratadas como fora da proposta.

## Instruções de entrega

A entrega deverá ser feita no GitHub ou no próprio Google Colab.

Links **sem permissão** de acesso terão um desconto de 20% na nota.